# tcrkit — repertoire metrics

These measure a *set* of TCRs. They need the optional extra:
`pip install 'tcrkit[metrics]'`.

| | answers |
|---|---|
| `diversity` | how evenly is a repertoire spread? |
| `morisita_horn` | how much do two repertoires overlap? |
| `generation_probability` | how likely was recombination to make this CDR3? (OLGA) |
| `tcrdist` | how similar are two TCRs? (tcrdist3) |

In [1]:
import pandas as pd

import tcrkit

reps = pd.DataFrame({
    'sample': ['expanded'] * 3 + ['even'] * 3,
    'cdr3':   ['A', 'B', 'C'] * 2,
    'count':  [90, 5, 5,   10, 10, 10],
})

## 1. `diversity`

One row per repertoire. `by=` names the column that splits them. `clonality` is 0 for a
perfectly even repertoire and 1 for a single clone.

In [2]:
tcrkit.diversity(reps, by='sample', col_mapping={'clone': 'cdr3'}).round(3)

,repertoire,n_clones,n_cells,shannon,pielou_evenness,clonality,simpson,dominance,top_clone_fraction
0,even,3,30.0,1.585,1.000,0.000,0.667,0.333,0.333
1,expanded,3,100.0,0.569,0.359,0.641,0.185,0.815,0.900


## 2. `morisita_horn`

The overlap is 0 when two repertoires share no clones and 1 when their clone frequencies
match. It ignores sequencing depth:

In [3]:
tcrkit.morisita_horn({'A': 10, 'B': 5}, {'A': 100, 'B': 50}, squeeze=True)

np.float64(1.0)

## 3. `generation_probability`

Pass the CDR3 *with* its conserved C and F, as `add_cdr3_junctions` returns it. V and J
are optional and narrow the result. Compare on `log10_pgen`:

In [4]:
tcrkit.generation_probability(['CASSIRSSYEQYF', 'CASSYSIRGSRGEQYF'], v='TRBV19', j='TRBJ2-7')[
    ['cdr3', 'log10_pgen', 'error']]

,cdr3,log10_pgen,error
0,CASSIRSSYEQYF,-7.319760,None
1,CASSYSIRGSRGEQYF,-14.078411,None


## 4. `tcrdist`

Columns are `{chain}_cdr3` / `{chain}_v` / `{chain}_j`. Identical TCRs are 0 apart.
`long=True` gives one row per pair instead of the square matrix.

In [5]:
tcrs = pd.DataFrame({
    'beta_cdr3': ['CASSIRSSYEQYF', 'CASSIRSSYEQYF', 'CASSLGQAYEQYF'],
    'beta_v':    ['TRBV19', 'TRBV19', 'TRBV20-1'],
    'beta_j':    ['TRBJ2-7'] * 3,
}, index=['t1', 't1_copy', 't2'])

tcrkit.tcrdist(tcrs)

,t1,t1_copy,t2
t1,0,0,99
t1_copy,0,0,99
t2,99,99,0


**Naming:** `simpson` here is 1 − Σp², and `dominance` is Σp². Some papers call Σp²
"Simpson". `clonality` is 1 − Pielou evenness, whereas immunarch uses the word for the top-clone share
(`top_clone_fraction` here).